## Exercise: Final EDA Workflow

You are a data scientist working for a financial services company. The company wants to predict which customers are likely to miss payment within the next 60 days. You receive a dataset containing:
- Customer demographics.
- Accounot balances.
- Historical payment records.
- Transaction activity.
- Credit utilization.
- Customer service interactions.
- Account status.
- Payment delinquency outcomes.

Your task is to design a complete exploratory data analysis and preprocessing workflow.

### Project Requirements

Chapter 5 Capstone Checklist
- Problem definition: Define the target, observation unit, prediction timestamp, and success metrics.
- Dataset inspection: Identify features, data types, identifiers, and structural issues.
- Data quality audit: Check missingness, duplicates, invalid values, and inconsistent categories.
- Univariate EDA: Analyze numerical and categorical distributions.
- Multivariate EDA: Investigate correlations, feature relationships, and interactions.
- Target analysis: Evaluate class imbalance and relationships with candidate predictors.
- Leakage audit: Identify unavailable future information and invalid feature construction.
- Evaluation design: Choose a train-validation-test strategy appropriate for 60-day outcomes.
- Preprocessing pipeline: Implement leakage-safe imputation, encoding, and scaling.
- Baseline modeling: Train a baseline model and evaluate appropriate metrics.
- Documentation: Summarize findings, assumptions, limitations, and recommended next steps.

### Expected Deliverables

A complete submission should include a reproducible Jupyter notebook, a documented preprocessing pipeline, appropriate evaluation results, and a written summary of findings.

A strong submission should explain not only what was done, but why each decision was appropriate for the business problem and data structure.

### Stage 1 — Problem Definition

Define the target, observation unit, prediction timestamp, and success metrics. The example dataset has been loaded below.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 5],
    "age": [25, 42, 300, 36, 51, 51],
    "income": [
        45000,
        np.nan,
        85000,
        62000,
        150000,
        150000
    ],
    "premium": [
        1200,
        1800,
        -500,
        2400,
        6000,
        6000
    ],
    "policy_type": [
        "Auto",
        "Home",
        "Auto",
        "Life",
        "auto",
        "auto"
    ],
    "cancelled": [0, 1, 0, 0, 1, 1]
})

print(df)

The problem we are trying to solve is to predict which customers are likely to miss a payment within the next 60 days. The target variable is whether they will miss or not represented as `1` and `0` or `True` and `False`. The observational unit are financial transactions. Customers could have multiple financial transactions so we could expect to see duplicates of customer IDs. The prediction timestamp is the next 60 days so we'll have to split the data according to the time of transactions to ensure no future or already miss payment transactions are included in the training data. The success metrics we'll use are accuracy, precision, recall, F1, PR-AUC. Accuracy can be used a baseline, precision and recall will be used to assess how accurate the predictions are with an unbalanced dataset since we can expect that most transactions aren't fraudulent. F1 can be used to assess the business tradeoff of prioritizing precision and recall, PR-AUC will be used to interpret how well the model predicts the minority positive class. 

### Stage 2 — Dataset Inspection

Identify features, data types, identifiers, and structural issues.

In [ ]:
print("Shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nFull metadata summary:")
print(df.info())

print("\nSummarize text and categorical columns:")
print(df.describe(include='object'))

print("\nCount category frequencies:")
print(df.value_counts())

### Stage 3 — Data Quality Audit

Check missingness, duplicates, invalid values, and inconsistent categories.

In [ ]:
print("Missing values:", df.isna().sum())

print("\nExact duplicates:")
print(df.duplicated().sum())

print("\nDuplicate IDs:")
print(
    df["customer_id"].duplicated().sum()
    df["transaction_id"].duplicated.sum()
    df["account_id"].duplicated.sum()
)

print("\nInvalid ages:")
print(
    df[
        ~df["age"].between(18, 100)
    ]
)

print("\nInvalid transactions:")
print(
    df[df["transaction"] <= 0]
)

print("\nInvalid balances:")
print(
    df[df["balance"] <= 0]
)

df_clean = df.drop_duplicates(subset=["transaction_id"])

df_clean["transaction_type"] = (
    df_clean["transaction_type"]
    .str.strip()
    .str.title()
)

df_clean["account_type"] = (
    df_clean["account_type"]
    .str.strip()
    .str.title()
)

df_clean["location"] = (
    df_clean["location"]
    .str.strip()
    .str.title()
)

df_clean["customer_name"] = (
    df_clean["customer_name"]
    .str.strip()
    .str.title()
)

df_clean.loc[
    ~df_clean["age"].between(18, 100)
    "age"
] = np.nan

df_clean.loc[
    ~df_clean["transaction"].between(0, np.inf),
    "transaction"
] = np.nan

df_clean.loc[
    ~df_clean["balance"].between(0, np.inf),
    "balance"
] = np.nan

print("\nPayment delinquincy outcomes:")
print(df_clean["payment_delinquincy_outcome"].mean())

print("\nSummary Statistics")
print(df_clean.describe(include="all"))

### Stage 4 — Univariate EDA 

Analyze numerical and categorical distributions.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numerical_features = df_clean.select_dtypes(include=['number']).columns
categorical_features = df_clean.select_dtypes(include=['object', 'category', 'bool']).columns

# Numerical Univariate Summary
for feature in numerical_features:
    print(f"{feature} (Skew: {df_clean[feature].skew():.2f})")
    print(df_clean[feature].describe(), "\n")

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    sns.hist(df_clean[feature], kde=True, ax=axes[0])
    sns.boxplot(x=df_clean[feature], ax=axes[1])
    plt.suptitle(col)
    plt.tight_layout()
    plt.show()

# Categorical Univariate Summary
for feature in categorical_features:
    print(f"{feature} (Unique: {df_clean[feature].nunique()})")
    print(df_clean[feature].value_counts(normalize=True).mul(100).round(2), "\n")

    plt.figure(figsize=(8, 3))
    sns.countplot(data=df_clean, y=feature, order=df[feature].value_counts().index[:10])
    plt.title(f"Top Categories in {feature}")
    plt.tight_layout()
    plt.show()

### Stage 5 — Multivariate EDA 

Investigate correlations, feature relationships, and interactions.

In [ ]:
# Numerical Correlation Heatmap
plt.figure(figsize=(10, 8))
corr = df_clean[numerical_features].corr(method='spearman')
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title("Spearman Correlation Matrix")
plt.tight_layout()
plt.show()

# Numerical vs. Categorical
target = df_clean['missed_payment']   # Replace with target column
for feature in categorical_features[:5]:
    print(f"\n{target} by {feature}")
    print(df_clean.groupby(feature)[target].agg(['count', 'mean', 'median', 'std']))

    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df_clean, x=feature, y=target)
    plt.title(f"{target} Distribution by {feature}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### Stage 6 — Target analysis 

Evaluate class imbalance and relationships with candidate predictors.

In [ ]:
# Target Balance Check
counts = df_clean[target].value_counts()
pcts = df_clean[target].value_counts(normalize=True) * 100
print("Target Distribution")
print(pd.DataFrame({'Count': counts, 'percentage (%)': pcts.round(2)}))
print(f"Imbalance Ratio: {counts.max() / counts.min():.2f}:1/n")

# Categorical Predictors vs. Target Rate
for feature in categorical_features:
    summary = df_clean.groupby(feature, observed=False)[target].agg(
        Count='count',
        Positive_Rate=lambda x: round(x.mean() * 100, 2)
    ).sort_values(by='Positive_Rate', ascending=False)
    print(f"Target Rate by {feature}")
    print(summary, "/n")

# Numerical Predictors vs. Target (Normalized KDE for Imbalanced Classes)
for feature in numerical_features[:5]:
    plt.figure(figsize=(7, 3))
    sns.kdeplot(data=df_clean, x=feature, hue=target, common_norm=False, fill=True)
    plt.title(f"{feature} Distribution by {target} (Normalized)")
    plt.tight_layout()
    plt.show()

# Quick Mutual Information Ranking (Numeric Features)
X_num = df_clean[numerical_features].fillna(df_clean[numerical_features].median())
mi_scores = pd.Series(
    mutual_info_classif(X_num, df_clean[target], random_state=42),
    index=numerical_features
).sort_values(ascending=False)
print("Mutual Information Ranking (Numeric)")
print(mi_scores.round(4))

### Stage 7 — Leakage Audit

Identify unavailable future information and invalid feature construction. 

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

# Single-Feature AUC Check (Flags suspiciously predictive features)
print("Single-Feature ROC-AUC Audit (Threshold > 0.90)")
for feature in numerical_features:
    valid = df_clean[[feature, target]].dropna()
    if valid[feature].nunique() > 1:
        auc = roc_auc_score(valid[target], valid[feature])
        auc = max(auc, 1 - auc)     # Handle inverse relationships
        if auc > 0.90:
            print(f"[WARNING] {feature}: Single-feature AUC = {auc:.4f}")

# Target-Correlated Missingness Check
print("\nMissingness Rate by Target Class")
missing_by_target = df_clean.isna().groupby(df_clean[target]).mean().T
missing_by_target['diff'] = (missing_by_target[1] - missing_by_target[0]).abs()
suspicious_nulls = missing_by_target[missing_by_target['diff'] > 0.20]
if not suspicious_nulls.empty:
    print(suspicious_nulls.sort_values(by='diff', ascending=False))

# Monotonic ID / Index Leakage Check
print("\n Correlation with Row Index")
index_corr = pd.Series(df.index, index=df.index).corr(df[target])
print(f"Row index vs. target correlation: {index_corr:.4f}")

### Stage 8 — Evaluation Design

Choose a train-validation-test strategy appropriate for 60-day outcomes.

In [ ]:
df_clean["payment_date"] = pd.to_datetime(df_clean["payment_date"])
df_clean = df_clean.sort_values("payment_date").reset_index(drop=True)

latest_valid_obs = df["payment_date"].max() - pd.Timedelta(days=60)
df_mature = df_clean[df_clean["payment_date"] <= latest_valid_obs].copy()

test_start = latest_valid_obs - pd.Timedelta(days=90)
val_end = test_start - pd.Timedelta(days=60)
val_start = val_end - pd.Timedelta(days=90)
train_end = val_start - pd.Timedelta(days=60)

train_df = df_mature[df_mature["payment_date"] <= train_end]
val_df = df_mature[(df_mature["payment_date"] >= val_start) & (df_mature["observed_date"] <= val_end)]
test_df = df_mature[df_mature["payment_date"] >= test_start]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2806961734.py, line 13)